In [8]:
#Load evaluation dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
val_df = pd.read_csv("/Users/rohanbabbar/Documents/Final Year/cwe/Archive/juliet_test_template_split.csv") 

#Create dataset class
import torch
from torch.utils.data import Dataset

class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]))
        }
#Create Dataloader
from torch.utils.data import DataLoader
MODEL_DIR = "/Users/rohanbabbar/Documents/Final Year/cwe/Archive/codet5_juliet_multiclass"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

print("✅ Model & tokenizer loaded")

dataset = JulietDataset(
    val_df["code_clean"],
    val_df["cwe_index"],
    tokenizer
)

loader = DataLoader(dataset, batch_size=8, shuffle=False)

✅ Model & tokenizer loaded


In [19]:
import pandas as pd

# Load original dataset
df = pd.read_csv("juliet_cwe_dataset.csv")

# Create mapping
cwe_mapping = (
    df[["cwe_index", "cwe"]]
    .drop_duplicates()
    .sort_values("cwe_index")
)

print(cwe_mapping.head(20))

        cwe_index     cwe
70409           0  CWE114
72189           1  CWE121
114929          2  CWE122
33575           3  CWE123
148033          4  CWE124
110773          5  CWE126
61899           6  CWE127
98135           7  CWE134
28293           8   CWE15
179683          9  CWE176
28137          10  CWE188
2380           11  CWE190
160229         12  CWE191
20013          13  CWE194
28891          14  CWE195
25063          15  CWE196
34885          16  CWE197
169619         17  CWE222
56135          18  CWE223
0              19  CWE226


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report



# Print classification report with CWE names
print(classification_report(true_cwe, pred_cwe, digits=4))

              precision    recall  f1-score   support

      CWE114     1.0000    0.9910    0.9955       444
      CWE121     1.0000    0.9971    0.9986      2780
      CWE122     1.0000    0.9978    0.9989      3614
      CWE124     0.9925    0.9987    0.9956      1598
      CWE126     1.0000    0.9979    0.9989       950
      CWE127     1.0000    0.9987    0.9994      1598
      CWE134     0.9971    1.0000    0.9985      1368
      CWE190     1.0000    0.9987    0.9994      3104
      CWE191     0.9931    0.9988    0.9960      1736
      CWE194     1.0000    0.9865    0.9932       740
      CWE195     1.0000    0.9838    0.9918       740
      CWE197     0.9136    1.0000    0.9548       296
       CWE23     1.0000    1.0000    1.0000      1254
      CWE244     1.0000    1.0000    1.0000        36
      CWE252     1.0000    1.0000    1.0000       144
      CWE253     1.0000    1.0000    1.0000       180
      CWE272     1.0000    1.0000    1.0000       216
      CWE284     1.0000    

In [27]:
all_preds, all_labels = [], []

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
model.eval()

with torch.no_grad():
    for batch in loader:
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        preds = outputs.logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

# ----------------------------
# Load CWE mapping
# ----------------------------
mapping_df = pd.read_csv("juliet_cwe_dataset.csv")[["cwe_index", "cwe"]].drop_duplicates()
cwe_map = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

# Convert numeric labels to CWE names
true_cwe = [cwe_map.get(i, f"CWE_{i}") for i in all_labels]
pred_cwe = [cwe_map.get(i, f"CWE_{i}") for i in all_preds]

# ----------------------------
# Classification report with CWE names
# ----------------------------
from sklearn.metrics import classification_report

print(classification_report(true_cwe, pred_cwe, digits=4))

              precision    recall  f1-score   support

      CWE114     1.0000    0.9910    0.9955       444
      CWE121     1.0000    0.9971    0.9986      2780
      CWE122     1.0000    0.9978    0.9989      3614
      CWE124     0.9925    0.9987    0.9956      1598
      CWE126     1.0000    0.9979    0.9989       950
      CWE127     1.0000    0.9987    0.9994      1598
      CWE134     0.9971    1.0000    0.9985      1368
      CWE190     1.0000    0.9987    0.9994      3104
      CWE191     0.9931    0.9988    0.9960      1736
      CWE194     1.0000    0.9865    0.9932       740
      CWE195     1.0000    0.9838    0.9918       740
      CWE197     0.9136    1.0000    0.9548       296
       CWE23     1.0000    1.0000    1.0000      1254
      CWE244     1.0000    1.0000    1.0000        36
      CWE252     1.0000    1.0000    1.0000       144
      CWE253     1.0000    1.0000    1.0000       180
      CWE272     1.0000    1.0000    1.0000       216
      CWE284     1.0000    

In [14]:
print("Unique validation classes:", sorted(val_df["cwe_index"].unique()))
print("Total validation classes:", val_df["cwe_index"].nunique())

Unique validation classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(16), np.int64(20), np.int64(22), np.int64(24), np.int64(25), np.int64(28), np.int64(30), np.int64(31), np.int64(33), np.int64(34), np.int64(37), np.int64(41), np.int64(42), np.int64(43), np.int64(45), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(55), np.int64(56), np.int64(60), np.int64(63), np.int64(69), np.int64(72), np.int64(78), np.int64(80), np.int64(81), np.int64(82), np.int64(83), np.int64(86), np.int64(97), np.int64(98), np.int64(100), np.int64(104), np.int64(105), np.int64(107), np.int64(108), np.int64(109), np.int64(110), np.int64(113), np.int64(115), np.int64(117)]
Total validation classes: 56


In [28]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# 1. Load mapping file (index → CWE)
mapping_df = pd.read_csv("juliet_cwe_dataset.csv")[["cwe_index", "cwe"]].drop_duplicates()
cwe_map = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

# 2. Generate confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# 3. Get labels in sorted order
labels_sorted = sorted(set(all_labels))

# 4. Map indices to CWE names
cwe_labels = [cwe_map.get(i, f"CWE_{i}") for i in labels_sorted]

# 5. Create dataframe with CWE names
cm_df = pd.DataFrame(cm, index=cwe_labels, columns=cwe_labels)

# 6. Save
cm_df.to_csv("confusion_matrix_cwe.csv")

print("✅ Confusion matrix saved with CWE names.")

✅ Confusion matrix saved with CWE names.


In [24]:
import numpy as np
import pandas as pd

# Load CWE mapping
mapping_df = pd.read_csv("juliet_cwe_dataset.csv")[["cwe_index", "cwe"]].drop_duplicates()
cwe_map = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

cm_array = np.array(cm)

confusions = []

for i in range(len(cm_array)):
    for j in range(len(cm_array)):
        if i != j and cm_array[i][j] > 0:
            true_cwe = cwe_map.get(i, f"CWE_{i}")
            pred_cwe = cwe_map.get(j, f"CWE_{j}")
            confusions.append((true_cwe, pred_cwe, cm_array[i][j]))

# Sort by most frequent confusion
confusions.sort(key=lambda x: x[2], reverse=True)

print("Top confused CWE pairs:")
for true_cwe, pred_cwe, count in confusions[:15]:
    print(f"{true_cwe} predicted as {pred_cwe}: {count} times")

Top confused CWE pairs:
CWE188 predicted as CWE190: 10 times
CWE176 predicted as CWE190: 8 times
CWE121 predicted as CWE123: 6 times
CWE122 predicted as CWE123: 6 times
CWE114 predicted as CWE127: 4 times
CWE121 predicted as CWE190: 2 times
CWE122 predicted as CWE190: 2 times
CWE123 predicted as CWE15: 2 times
CWE124 predicted as CWE190: 2 times
CWE126 predicted as CWE15: 2 times
CWE134 predicted as CWE15: 2 times
CWE134 predicted as CWE190: 2 times
CWE15 predicted as CWE190: 2 times
CWE176 predicted as CWE15: 2 times
CWE188 predicted as CWE15: 2 times


In [12]:
eval_df = pd.DataFrame({
    "true": all_labels,
    "pred": all_preds
})

eval_df["correct"] = eval_df["true"] == eval_df["pred"]

# Error types
false_positive = eval_df[(eval_df["true"] != eval_df["pred"])]
false_negative = false_positive  # same set in multi-class

eval_df.to_csv("prediction_analysis.csv", index=False)

print("Total samples:", len(eval_df))
print("Correct:", eval_df["correct"].sum())
print("Errors:", len(false_positive))

Total samples: 35730
Correct: 35674
Errors: 56


In [13]:
error_counts = eval_df[~eval_df["correct"]]["true"].value_counts()

print("Worst performing CWEs:")
print(error_counts.head(20))

Worst performing CWEs:
true
14    12
13    10
1      8
2      8
11     4
0      4
41     2
6      2
5      2
4      2
12     2
Name: count, dtype: int64


In [26]:
import pandas as pd

# 1. Load confusion matrix (indices)
cm = pd.read_csv("confusion_matrix.csv", index_col=0)

# Convert index/columns to int
cm.index = cm.index.astype(int)
cm.columns = cm.columns.astype(int)

# 2. Load mapping dataset
mapping_df = pd.read_csv("juliet_cwe_dataset.csv")[["cwe_index", "cwe"]].drop_duplicates()

# Create mapping dictionary
cwe_map = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

# Replace numeric indices with CWE names
cm.index = cm.index.map(cwe_map)
cm.columns = cm.columns.map(cwe_map)

# 3. Extract confusion pairs
confusions = []

for true_cwe in cm.index:
    for pred_cwe in cm.columns:
        if true_cwe != pred_cwe:
            count = cm.loc[true_cwe, pred_cwe]
            if count > 0:
                confusions.append((true_cwe, pred_cwe, count))

confusions_df = pd.DataFrame(
    confusions,
    columns=["True_CWE", "Predicted_CWE", "Count"]
).sort_values("Count", ascending=False)

print(confusions_df.head(20))

   True_CWE Predicted_CWE  Count
14   CWE188        CWE190     10
12   CWE176        CWE190      8
1    CWE121        CWE123      6
3    CWE122        CWE123      6
0    CWE114        CWE127      4
2    CWE121        CWE190      2
4    CWE122        CWE190      2
5    CWE123         CWE15      2
6    CWE124        CWE190      2
7    CWE126         CWE15      2
8    CWE134         CWE15      2
9    CWE134        CWE190      2
10    CWE15        CWE190      2
11   CWE176         CWE15      2
13   CWE188         CWE15      2
15   CWE244         CWE15      2


In [16]:
similar_groups = {
    "Buffer Issues": ["CWE121","CWE122","CWE124"],
    "Integer Errors": ["CWE190","CWE191","CWE194","CWE195"],
    "Resource Errors": ["CWE404","CWE415","CWE416"],
    "Path Traversal": ["CWE23","CWE36"]
}

def find_group(cwe):
    for group, items in similar_groups.items():
        if cwe in items:
            return group
    return "Other"

confusions_df["True_Group"] = confusions_df["True_CWE"].apply(find_group)
confusions_df["Pred_Group"] = confusions_df["Predicted_CWE"].apply(find_group)

semantic_confusions = confusions_df[
    confusions_df["True_Group"] == confusions_df["Pred_Group"]
]

print(semantic_confusions.head())

    True_CWE Predicted_CWE  Count True_Group Pred_Group
5          2             2   3606      Other      Other
15         7             7   3100      Other      Other
68        52            52   2960      Other      Other
2          1             1   2772      Other      Other
65        49            49   1872      Other      Other


In [18]:
support_df = pd.read_csv("classification_report_updated.csv")
# Convert numeric index → CWE format
confusions_df["True_CWE"] = "CWE" + confusions_df["True_CWE"].astype(str)
confusions_df["Predicted_CWE"] = "CWE" + confusions_df["Predicted_CWE"].astype(str)
confusions_df = confusions_df.merge(
    support_df[["label","support"]],
    left_on="True_CWE",
    right_on="label",
    how="left"
).rename(columns={"support":"true_support"})

confusions_df = confusions_df.merge(
    support_df[["label","support"]],
    left_on="Predicted_CWE",
    right_on="label",
    how="left"
).rename(columns={"support":"pred_support"})

imbalance_errors = confusions_df[
    confusions_df["true_support"] < confusions_df["pred_support"]
]

print(imbalance_errors.head())

Empty DataFrame
Columns: [True_CWE, Predicted_CWE, Count, True_Group, Pred_Group, label_x, true_support, label_y, pred_support]
Index: []


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Comparison")
plt.legend()
plt.show()